# 02 — Review Collection and Cleaning

Collect reviews for the ElectroLens product catalog using Amazon Reviews 2023 Electronics. Keep up to 50 reviews per product (up to 10 helpful and the rest sampled), clean duplicates, and save the results to Google Drive. A checkpoint lets the scan resume after interruptions.
Selection: up to 10 highest helpful_vote reviews, then random non-overlapping reviews to fill up to 50 total per product. Products with fewer reviews retain the available reviews (minimum 10 after cleaning).


In [1]:
%pip -q install pandas pyarrow tqdm requests

In [2]:
import os, re, json, heapq, pickle, random, html, hashlib, time
import requests
import pandas as pd
from tqdm.auto import tqdm

In [3]:
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/ElectroLens"
os.makedirs(ROOT, exist_ok=True)

Mounted at /content/drive


In [4]:
BASE = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"
REVIEWS_URL = f"{BASE}/review_categories/Electronics.jsonl"
PRODUCTS_PATH = f"{ROOT}/products.parquet"
CKPT_PATH = f"{ROOT}/reviews_checkpoint_50.pkl"
TOP_HELPFUL, TOTAL_PER_PRODUCT = 10, 50
MIN_TEXT_LEN, MIN_REVIEWS_KEEP = 15, 10
CKPT_EVERY, SEED = 1_000_000, 42

products = pd.read_parquet(PRODUCTS_PATH)
if products.empty or products.parent_asin.isna().any() or not products.parent_asin.is_unique:
    raise ValueError("Catalog must contain unique, non-null parent_asin values.")
wanted = set(products.parent_asin)
config = {"catalog_hash": hashlib.sha256("\n".join(sorted(wanted)).encode()).hexdigest(),
          "top_helpful": TOP_HELPFUL, "total": TOTAL_PER_PRODUCT, "min_text_len": MIN_TEXT_LEN,
          "source": REVIEWS_URL}
print(f"Catalog: {len(wanted):,} products")

Catalog: 18,249 products


In [5]:
def stream_lines(url, start=0, retries=8):

    offset = start
    attempts = 0
    while attempts < retries:
        try:
            headers = {"Accept-Encoding": "identity"}
            if offset:
                headers["Range"] = f"bytes={offset}-"
            with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as response:
                response.raise_for_status()
                if offset and (response.status_code != 206 or
                    not response.headers.get("Content-Range", "").startswith(f"bytes {offset}-")):
                    raise RuntimeError("Server does not support byte-range resume; scan cannot safely continue.")
                if response.headers.get("Content-Encoding", "identity").lower() not in ("identity", ""):
                    raise RuntimeError("Unexpected compressed response; byte offsets would be invalid.")
                buffer = b""
                for chunk in response.iter_content(chunk_size=1 << 20):
                    if not chunk:
                        continue
                    buffer += chunk
                    parts = buffer.split(b"\n")
                    buffer = parts.pop()
                    for line in parts:
                        offset += len(line) + 1
                        if line:
                            yield line, offset
                if buffer:

                    offset += len(buffer)
                    yield buffer, offset
            return
        except requests.exceptions.RequestException as exc:
            attempts += 1
            print(f"Connection interrupted at byte {offset:,}; retry {attempts}/{retries}: {exc}")
            time.sleep(min(attempts * 2, 20))
    raise RuntimeError("Review stream failed after retries.")

MARK_A = b'"parent_asin": "'
MARK_B = b'"parent_asin":"'

def get_asin(line):
    for marker in (MARK_A, MARK_B):
        i = line.find(marker)
        if i != -1:
            start = i + len(marker)
            end = line.find(b'"', start)
            if end != -1:
                return line[start:end].decode("utf-8", "ignore")
    return None

def clean_text(value):
    text = html.unescape(value if isinstance(value, str) else "")
    text = re.sub(r"<br\b[^>]*>", " ", text, flags=re.I)
    text = re.sub(r"</?[A-Za-z][^>]*>", " ", text)
    text = re.sub(r"\[\[(?:VIDEOID|ASIN):[^\]]*\]\]", " ", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip()

In [6]:
print("Review source:", REVIEWS_URL)
print("Checkpoint:", CKPT_PATH)
print("Maximum reviews per product:", TOTAL_PER_PRODUCT)

Review source: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Electronics.jsonl
Checkpoint: /content/drive/MyDrive/ElectroLens/reviews_checkpoint_50.pkl
Maximum reviews per product: 50


In [7]:
random.seed(SEED)
state = {}
uid = lines = offset = 0
completed = False

if os.path.exists(CKPT_PATH):

    with open(CKPT_PATH, "rb") as f:
        checkpoint = pickle.load(f)
    if checkpoint.get("config") != config:
        raise ValueError("Checkpoint belongs to a different catalog/configuration. Move it aside before a new scan.")
    state = checkpoint["state"]
    uid, lines, offset = checkpoint["uid"], checkpoint["lines"], checkpoint["offset"]
    completed = checkpoint.get("completed", False)
    random.setstate(checkpoint["rng"])
    print(f"Resuming at {lines:,} source rows")
else:
    print("Starting new review scan")

def save_ckpt():
    temp = CKPT_PATH + ".tmp"
    with open(temp, "wb") as f:
        pickle.dump({"config": config, "state": state, "uid": uid,
                     "lines": lines, "offset": offset, "rng": random.getstate(),
                     "completed": completed}, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp, CKPT_PATH)

if not completed:
    pbar = tqdm(initial=lines, desc="Review rows", unit="rows")
    last_ckpt = lines
    try:
        for line, next_offset in stream_lines(REVIEWS_URL, start=offset):
            asin = get_asin(line)
            if asin is None or asin in wanted:
                record = json.loads(line)
                asin = record.get("parent_asin")
                if asin in wanted:
                    text = clean_text(record.get("text"))
                    if len(text) >= MIN_TEXT_LEN:
                        uid += 1
                        helpful = int(record.get("helpful_vote") or 0)
                        review = (record.get("rating"), clean_text(record.get("title")), text,
                                  helpful, bool(record.get("verified_purchase")), record.get("timestamp"))
                        sample = state.setdefault(asin, [0, [], []])
                        sample[0] += 1
                        if len(sample[1]) < TOTAL_PER_PRODUCT:
                            sample[1].append((uid, review))
                        else:
                            j = random.randrange(sample[0])
                            if j < TOTAL_PER_PRODUCT:
                                sample[1][j] = (uid, review)
                        if helpful > 0:
                            item = (helpful, uid, review)
                            if len(sample[2]) < TOP_HELPFUL:
                                heapq.heappush(sample[2], item)
                            elif item[:2] > sample[2][0][:2]:
                                heapq.heapreplace(sample[2], item)
            lines += 1
            offset = next_offset
            pbar.update(1)
            if lines - last_ckpt >= CKPT_EVERY:
                save_ckpt()
                last_ckpt = lines
        completed = True
    finally:
        save_ckpt()
        pbar.close()
else:
    print("Scan already complete; using checkpoint")
print(f"Processed {lines:,} rows; matched products: {len(state):,}")

Starting new review scan


Review rows: 0rows [00:00, ?rows/s]

Processed 43,886,944 rows; matched products: 18,202


In [8]:
rows = []
selection_rng = random.Random(SEED)
for asin, (n_seen, reservoir, heap) in state.items():
    if asin not in wanted:
        continue
    top = sorted(heap, key=lambda item: (-item[0], -item[1]))
    top_ids = {review_id for _, review_id, _ in top}
    remaining = [(review_id, review) for review_id, review in reservoir if review_id not in top_ids]
    selection_rng.shuffle(remaining)

    slots = TOTAL_PER_PRODUCT - len(top)
    candidates = [(review, True) for _, _, review in top]
    candidates += [(review, False) for _, review in remaining[:slots]]
    for review, is_top in candidates:
        rows.append({"parent_asin": asin, "rating": review[0], "title": review[1],
                     "text": review[2], "helpful_vote": review[3],
                     "verified_purchase": review[4], "timestamp": review[5],
                     "is_top_helpful": is_top, "n_text_reviews_seen": n_seen})
if not rows:
    raise RuntimeError("No matching reviews were found.")
reviews = pd.DataFrame(rows)
reviews["title"] = reviews["title"].map(clean_text)
reviews["text"] = reviews["text"].map(clean_text)
reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")
reviews["timestamp"] = pd.to_numeric(reviews["timestamp"], errors="coerce")
reviews = reviews[reviews["text"].str.len().ge(MIN_TEXT_LEN)
                  & reviews["rating"].isin([1, 2, 3, 4, 5])
                  & reviews["timestamp"].ge(0)].copy()
duplicate_columns = ["parent_asin", "rating", "title", "text", "timestamp"]
before = len(reviews)
reviews = reviews.drop_duplicates(duplicate_columns, keep="first")
reviews = reviews.groupby("parent_asin", sort=False).head(TOTAL_PER_PRODUCT).reset_index(drop=True)
counts = reviews.groupby("parent_asin").size()
keep_ids = set(counts[counts >= MIN_REVIEWS_KEEP].index)
if not keep_ids:
    raise RuntimeError("No products have enough cleaned reviews.")
print(f"Removed {before - len(reviews):,} duplicate reviews")
print(f"Products with >= {MIN_REVIEWS_KEEP} reviews: {len(keep_ids):,}/{len(wanted):,}")

Removed 3,968 duplicate reviews
Products with >= 10 reviews: 13,328/18,249


In [9]:
if not completed:
    raise RuntimeError("Review scan is incomplete. Resume Cell 7 before saving final results.")
products_final = products[products.parent_asin.isin(keep_ids)].reset_index(drop=True)
reviews_final = reviews[reviews.parent_asin.isin(keep_ids)].reset_index(drop=True)
reviews_final.insert(0, "review_id", range(len(reviews_final)))
assert set(reviews_final.parent_asin) == set(products_final.parent_asin)
assert reviews_final.groupby("parent_asin").size().between(MIN_REVIEWS_KEEP, TOTAL_PER_PRODUCT).all()
assert not reviews_final.duplicated(duplicate_columns).any()
products_final.to_parquet(f"{ROOT}/products_final.parquet", index=False)
reviews_final.to_parquet(f"{ROOT}/reviews.parquet", index=False)
print(f"Saved {len(products_final):,} products and {len(reviews_final):,} reviews to {ROOT}")

Saved 13,328 products and 506,849 reviews to /content/drive/MyDrive/ElectroLens
